# Engenharia de Atributos

A análise exploratória revelou diversas variáveis fortemente associadas aos preços dos imóveis, assim como grupos de variáveis que descrevem aspectos relacionados de uma propriedade. Neste notebook, utilizamos essas informações para criar novos atributos que podem ajudar os modelos de aprendizado de máquina a capturar padrões importantes presentes nos dados.

A engenharia de atributos é o processo de transformar variáveis existentes em novas representações que reflitam melhor os fatores subjacentes que influenciam a variável alvo. Ao combinar variáveis relacionadas, agregar informações e criar indicadores relevantes, podemos fornecer aos modelos uma visão mais informativa do problema do que aquela disponível apenas nos dados originais.

Os atributos introduzidos neste notebook são motivados por conhecimentos básicos do domínio de imóveis residenciais. De modo geral, compradores consideram fatores como área total habitável, idade do imóvel, qualidade geral da construção, comodidades disponíveis e quantidade de espaço útil destinado a diferentes finalidades. Os atributos construídos abaixo buscam capturar esses conceitos de forma mais direta.

Antes de criar novos atributos, primeiro carregamos o conjunto de dados processado gerado no notebook anterior. Esse conjunto já inclui o tratamento dos valores faltantes e as correções de qualidade dos dados identificadas durante a etapa de análise exploratória.

Em seguida, importamos as bibliotecas necessárias para a engenharia de atributos e para as etapas posteriores de pré-processamento.


In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

sns.set_theme()

# Load data processed and saved in the EDA notebook
df_full = pd.read_parquet(
    "../data/processed/01_data.parquet"
)
df_features = pd.read_parquet(
    "../data/processed/01_features.parquet"
)

In [2]:
df_full

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,IsTrainSet
0,1,60,RL,65.0,8450,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,2,2008,WD,Normal,208500.0,True
1,2,20,RL,80.0,9600,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,5,2007,WD,Normal,181500.0,True
2,3,60,RL,68.0,11250,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,9,2008,WD,Normal,223500.0,True
3,4,70,RL,60.0,9550,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,2,2006,WD,Abnorml,140000.0,True
4,5,60,RL,84.0,14260,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,12,2008,WD,Normal,250000.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2915,160,RM,21.0,1936,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,6,2006,WD,Normal,NaN,False
2915,2916,160,RM,21.0,1894,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,4,2006,WD,Abnorml,NaN,False
2916,2917,20,RL,160.0,20000,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,9,2006,WD,Abnorml,NaN,False
2917,2918,85,RL,62.0,10441,Pave,None,Reg,Lvl,AllPub,...,None,MnPrv,Shed,700,7,2006,WD,Normal,NaN,False


In [3]:
df_features

,Name,Type,Category,Source,Description
0,MSSubClass,numerical,original,MSSubClass,Identifies the type of dwelling involved in th...
1,MSZoning,categorical,original,MSZoning,Identifies the general zoning classification o...
2,LotFrontage,numerical,original,LotFrontage,Linear feet of street connected to property
3,LotArea,numerical,original,LotArea,Lot size in square feet
4,Street,categorical,original,Street,Type of road access to property
...,...,...,...,...,...
74,MiscVal,numerical,original,MiscVal,$Value of miscellaneous feature
75,MoSold,numerical,original,MoSold,Month Sold (MM)
76,YrSold,numerical,original,YrSold,Year Sold (YYYY)
77,SaleType,categorical,original,SaleType,Type of sale


## Atributos agregados construídos

Atributos agregados combinam informações de múltiplas variáveis em uma única medida. O objetivo é capturar características mais amplas dos imóveis que podem ser mais informativas do que seus componentes individuais analisados separadamente. Por exemplo, a área total habitável ou o número total de banheiros podem fornecer uma descrição mais completa de uma propriedade do que considerar cada variável contribuinte de forma isolada.


### TotalRooms

O número total de cômodos geralmente é mais informativo do que a quantidade individual de cada tipo de cômodo analisada separadamente. Este atributo busca capturar a capacidade funcional geral do imóvel.


In [4]:
# Create the feature
df_full["TotalRooms"] = (
    df_full["TotRmsAbvGrd"]
    + df_full["KitchenAbvGr"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalRooms",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "TotRmsAbvGrd;KitchenAbvGr",
    "Description": "Total number of living spaces and kitchens"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### TotalBathrooms

O conjunto de dados distribui as informações sobre banheiros em diversas variáveis. Combiná-las em uma única medida fornece uma representação mais completa das instalações sanitárias do imóvel.


In [5]:
# Create the feature
df_full["TotalBathrooms"] = (
    df_full["FullBath"]
    + 0.5*df_full["HalfBath"]
    + df_full["BsmtFullBath"]
    + 0.5*df_full["BsmtHalfBath"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalBathrooms",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "FullBath;HalfBath;BsmtFullBath;BsmtHalfBath",
    "Description": "Total number of bathrooms, accounting for half bathrooms"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### TotalArea

O conjunto de dados original distribui as informações de área habitável em diversas variáveis correspondentes aos diferentes níveis do imóvel. Embora essas informações sejam úteis, compradores geralmente estão interessados na quantidade total de espaço utilizável disponível. Este atributo agrega as principais áreas habitáveis em uma única medida do tamanho do imóvel.


In [6]:
# Create the feature
df_full["TotalArea"] = (
    df_full["TotalBsmtSF"]
    + df_full["1stFlrSF"]
    + df_full["2ndFlrSF"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalArea",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "TotalBsmtSF;1stFlrSF;2ndFlrSF",
    "Description": "Combined usable area of the house"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### TotalPorchArea

As áreas de varanda são representadas por diversas variáveis separadas, dependendo do seu tipo. Este atributo agrega essas informações em uma única medida de espaço externo utilizável.


In [7]:
# Create the feature
df_full["TotalPorchArea"] = (
    df_full["OpenPorchSF"]
    + df_full["EnclosedPorch"]
    + df_full["3SsnPorch"]
    + df_full["ScreenPorch"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalPorchArea",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "OpenPorchSF;EnclosedPorch;3SsnPorch;ScreenPorch",
    "Description": "Combined area of all porch spaces"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### TotalQualityScore

As avaliações de qualidade geral e condição geral descrevem aspectos complementares de um imóvel. A combinação dessas variáveis cria uma medida mais ampla da qualidade geral da propriedade.


In [8]:
# Create the feature
df_full["TotalQualityScore"] = (
    df_full["OverallQual"]
    + df_full["OverallCond"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalQualityScore",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "OverallQual;OverallCond",
    "Description": "Combined measure of overall quality and condition"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### LuxuryFeatureCount

Este atributo contabiliza o número de determinadas comodidades presentes em um imóvel. Embora cada uma dessas características forneça informações individualmente, combiná-las em uma única variável cria uma medida simples do nível de comodidades oferecidas pela propriedade.

O termo *luxury* é utilizado aqui de forma ampla para se referir a características frequentemente associadas a imóveis maiores ou de maior valor. Valores mais altos indicam que uma propriedade possui um número maior dessas comodidades e, portanto, pode estar associada a um preço de mercado mais elevado.


In [9]:
# Create the feature
df_full["LuxuryFeatureCount"] = (
    (df_full["PoolArea"] > 0).astype(int)
    + (df_full["GarageArea"] > 0).astype(int)
    + (df_full["TotalBsmtSF"] > 0).astype(int)
    + (df_full["Fireplaces"] > 0).astype(int)
)

# Add feature to df_features
new_row = {
    "Name": "LuxuryFeatureCount",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "HasPool;HasGarage;HasBasement;HasFireplace",
    "Description": "Count of selected premium property amenities"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Atributos de diferença construídos

Atributos de diferença representam o tempo decorrido entre dois eventos ou medidas relacionadas. Em dados imobiliários, a idade de um imóvel, garagem ou reforma pode ser mais informativa do que os respectivos anos de ocorrência, pois compradores geralmente consideram o quão antigo ou recentemente atualizado é um imóvel, em vez das datas do calendário envolvidas.


### HouseAge

A idade de um imóvel no momento da venda provavelmente influencia seu valor de mercado. Casas mais novas frequentemente apresentam preços mais elevados devido aos padrões construtivos modernos, menores necessidades de manutenção e características mais atualizadas.


In [10]:
# Create the feature
df_full["HouseAge"] = (
    df_full["YrSold"]
    - df_full["YearBuilt"]
)

# Add feature to df_features
new_row = {
    "Name": "HouseAge",
    "Type": "numerical",
    "Category": "difference",
    "Source": "YrSold;YearBuilt",
    "Description": "Age of the property at the time of sale"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### YearsSinceRemodel

O ano original da reforma fornece informações úteis, mas o número de anos desde a última renovação pode estar mais diretamente relacionado à percepção de conservação do imóvel e ao seu valor de mercado.


In [11]:
# Create the feature
df_full["YearsSinceRemodel"] = (
    df_full["YrSold"]
    - df_full["YearRemodAdd"]
)

# Add feature to df_features
new_row = {
    "Name": "YearsSinceRemodel",
    "Type": "numerical",
    "Category": "difference",
    "Source": "YrSold;YearRemodAdd",
    "Description": "Years elapsed since the most recent remodeling"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### GarageAge

Uma garagem mais nova pode ser mais desejável do que uma garagem antiga devido ao menor desgaste e aos padrões construtivos mais modernos. Este atributo captura a idade da garagem no momento da venda do imóvel.


In [12]:
# Create the feature
df_full["GarageAge"] = (
    df_full["YrSold"]
    - df_full["GarageYrBlt"]
)

# Add feature to df_features
new_row = {
    "Name": "GarageAge",
    "Type": "numerical",
    "Category": "difference",
    "Source": "YrSold;GarageYrBlt",
    "Description": "Age of the garage at the time of sale"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Atributos de interação construídos

Atributos de interação combinam duas ou mais variáveis de forma que seus efeitos conjuntos possam ser representados explicitamente. Esses atributos podem ajudar a capturar relações que não são evidentes quando cada variável é analisada individualmente. Por exemplo, uma casa grande pode ter um valor elevado, e uma casa de alta qualidade também pode apresentar um preço maior. No entanto, um imóvel que seja simultaneamente grande e de alta qualidade pode alcançar um valor de mercado ainda mais elevado.


### QualityArea

A análise anterior mostrou que tanto a área habitável quanto a qualidade geral do imóvel apresentam forte associação com o preço de venda. Este atributo captura a interação entre essas duas variáveis, permitindo que o modelo diferencie imóveis que são grandes, de alta qualidade ou que possuem ambas as características. Ele pode ser interpretado como uma medida aproximada de área habitável ajustada pela qualidade do imóvel.


In [13]:
# Create the feature
df_full["QualityArea"] = (
    df_full["OverallQual"]
    * df_full["GrLivArea"]
)

# Add feature to df_features
new_row = {
    "Name": "QualityArea",
    "Type": "numerical",
    "Category": "interaction",
    "Source": "OverallQual;GrLivArea",
    "Description": "Interaction between house quality and living area"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Atributos de razão construídos

Atributos de razão descrevem a relação entre duas quantidades, em vez de seus valores absolutos. Esses atributos podem fornecer uma representação mais significativa de determinadas características dos imóveis, como a quantidade de espaço habitável por cômodo ou a área de garagem disponível por vaga de estacionamento.


### GarageAreaPerCar

A área total da garagem e o número de vagas disponíveis fornecem informações complementares sobre a garagem. Ao dividir a área pelo número de vagas, obtemos uma medida aproximada do espaço disponível por veículo.

Este atributo é um exemplo de transformação baseada em razão, que pode revelar relações que não são evidentes quando analisamos apenas as variáveis originais. Valores maiores podem indicar garagens mais espaçosas, com espaço adicional para armazenamento ou outras finalidades.

Ao construir este atributo, é necessário tomar cuidado para evitar divisões por zero. Imóveis sem garagem possuem `GarageCars = 0`, portanto a razão é definida como `0` para essas observações.


In [14]:
# Create the feature
df_full["GarageAreaPerCar"] = np.where(
    df_full["GarageCars"] > 0,
    df_full["GarageArea"] / df_full["GarageCars"],
    0
)

# Add feature to df_features
new_row = {
    "Name": "GarageAreaPerCar",
    "Type": "numerical",
    "Category": "ratio",
    "Source": "GarageArea;GarageCars",
    "Description": "Average garage area available per parking space"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### LivingAreaPerRoom

A área habitável total acima do solo e o número de cômodos fornecem informações importantes sobre o tamanho de uma casa. No entanto, essas variáveis descrevem aspectos diferentes da propriedade. Dois imóveis podem possuir a mesma área habitável, mas apresentar distribuições de espaço interno bastante distintas.

Ao dividir a área habitável pelo número de cômodos, obtemos uma medida aproximada do tamanho médio dos cômodos. Essa razão pode ajudar a diferenciar casas com muitos cômodos pequenos daquelas com menos cômodos, porém mais espaçosos.

Assim como em `GarageAreaPerCar`, ou em qualquer atributo baseado em razões, é necessário tomar cuidado para evitar divisões por zero. Neste conjunto de dados, todos os imóveis possuem pelo menos um cômodo acima do solo, mas ainda assim definimos a razão como `0` quando `TotRmsAbvGrd` for igual a zero, garantindo maior robustez na construção do atributo.


In [15]:
# Create the feature
df_full["LivingAreaPerRoom"] = np.where(
    df_full["TotRmsAbvGrd"] > 0,
    df_full["GrLivArea"] / df_full["TotRmsAbvGrd"],
    0
)

# Add feature to df_features
new_row = {
    "Name": "LivingAreaPerRoom",
    "Type": "numerical",
    "Category": "ratio",
    "Source": "GrLivArea;TotRmsAbvGrd",
    "Description": "Average living area per room"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Atributos indicadores binários construídos

Diversas variáveis descrevem naturalmente a presença ou ausência de uma determinada característica do imóvel. Embora essas variáveis possam ser representadas como valores booleanos, convertê-las para inteiros (`0` ou `1`) fornece um formato diretamente compatível com a maioria dos algoritmos de aprendizado de máquina e simplifica as etapas posteriores de pré-processamento.


### IsRemodeled

Indica se o imóvel passou por alguma reforma desde sua construção original.


In [16]:
# Create the feature
df_full["IsRemodeled"] = (
    df_full["YearBuilt"] != df_full["YearRemodAdd"]
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "IsRemodeled",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "YearBuilt;YearRemodAdd",
    "Description": "Indicates whether the property has been remodeled"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### HasPool

Indica se o imóvel possui piscina.


In [17]:
# Create the feature
df_full["HasPool"] = (
    df_full["PoolArea"] > 0
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "HasPool",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "PoolArea",
    "Description": "Indicates whether the property has a pool"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### HasGarage

Indica se o imóvel possui garagem.

In [18]:
# Create the feature
df_full["HasGarage"] = (
    df_full["GarageArea"] > 0
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "HasGarage",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "YearBuilt;YearRemodAdd",
    "Description": "Indicates whether the property has a garage"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### HasBasement

Indica se o imóvel possui porão.


In [19]:
# Create the feature
df_full["HasBasement"] = (
    df_full["TotalBsmtSF"] > 0
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "HasBasement",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "TotalBsmtSF",
    "Description": "Indicates whether the property has a basement"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### HasFireplace

Indica se o imóvel possui pelo menos uma lareira.

In [20]:
# Create the feature
df_full["HasFireplace"] = (
    df_full["Fireplaces"] > 0
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "HasFireplace",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "YearBuilt;YearRemodAdd",
    "Description": "Indicates whether the property has at least one fireplace"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Resumo dos atributos construídos

Ao longo deste notebook, criamos diversos novos atributos projetados para capturar informações que não estão diretamente disponíveis no conjunto de dados original. Esses atributos pertencem a diferentes categorias, incluindo atributos agregados, atributos baseados em tempo, atributos de razão, atributos de interação e indicadores binários.

Para facilitar o acompanhamento e a documentação, as informações sobre cada atributo são armazenadas na tabela de metadados `df_features`. Essa tabela registra o tipo do atributo, sua categoria, as variáveis de origem utilizadas em sua construção e uma breve descrição de seu significado.

As tabelas a seguir resumem os atributos construídos introduzidos neste notebook.


### Atributos agregados

Atributos agregados combinam informações de múltiplas variáveis em uma única medida.


In [21]:
df_features.loc[
    df_features["Category"].eq("aggregate")
]

,Name,Type,Category,Source,Description
79,TotalRooms,numerical,aggregate,TotRmsAbvGrd;KitchenAbvGr,Total number of living spaces and kitchens
80,TotalBathrooms,numerical,aggregate,FullBath;HalfBath;BsmtFullBath;BsmtHalfBath,"Total number of bathrooms, accounting for half..."
81,TotalArea,numerical,aggregate,TotalBsmtSF;1stFlrSF;2ndFlrSF,Combined usable area of the house
82,TotalPorchArea,numerical,aggregate,OpenPorchSF;EnclosedPorch;3SsnPorch;ScreenPorch,Combined area of all porch spaces
83,TotalQualityScore,numerical,aggregate,OverallQual;OverallCond,Combined measure of overall quality and condition
84,LuxuryFeatureCount,numerical,aggregate,HasPool;HasGarage;HasBasement;HasFireplace,Count of selected premium property amenities


### Atributos de diferença

Atributos de diferença capturam o tempo decorrido ou a distância entre duas quantidades relacionadas.


In [22]:
df_features.loc[
    df_features["Category"].eq("difference")
]

,Name,Type,Category,Source,Description
85,HouseAge,numerical,difference,YrSold;YearBuilt,Age of the property at the time of sale
86,YearsSinceRemodel,numerical,difference,YrSold;YearRemodAdd,Years elapsed since the most recent remodeling
87,GarageAge,numerical,difference,YrSold;GarageYrBlt,Age of the garage at the time of sale


### Atributos de razão

Atributos de razão descrevem a relação entre duas quantidades e frequentemente fornecem uma medida mais informativa do que qualquer uma das variáveis analisadas individualmente.


In [23]:
df_features.loc[
    df_features["Category"].eq("ratio")
]

,Name,Type,Category,Source,Description
89,GarageAreaPerCar,numerical,ratio,GarageArea;GarageCars,Average garage area available per parking space
90,LivingAreaPerRoom,numerical,ratio,GrLivArea;TotRmsAbvGrd,Average living area per room


### Atributos indicadores

Atributos indicadores representam explicitamente a presença ou ausência de características importantes de um imóvel.


In [24]:
df_features.loc[
    df_features["Category"].eq("indicator")
]

,Name,Type,Category,Source,Description
91,IsRemodeled,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has been remodeled
92,HasPool,numerical,indicator,PoolArea,Indicates whether the property has a pool
93,HasGarage,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has a garage
94,HasBasement,numerical,indicator,TotalBsmtSF,Indicates whether the property has a basement
95,HasFireplace,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has at least on...


## Análise de assimetria das variáveis

Muitos algoritmos de aprendizado de máquina apresentam melhor desempenho quando as variáveis numéricas possuem distribuições mais próximas de uma distribuição simétrica. Variáveis com alta assimetria podem dar importância desproporcional a observações extremas e dificultar que alguns modelos aprendam relações significativas nos dados.

Para identificar variáveis potencialmente problemáticas, calculamos a assimetria (*skewness*) de cada variável numérica. Uma assimetria positiva indica uma cauda longa à direita, enquanto uma assimetria negativa indica uma cauda longa à esquerda. Variáveis cuja assimetria ultrapassa um limite predefinido serão transformadas utilizando uma transformação logarítmica.

O objetivo não é forçar todas as variáveis a apresentar uma distribuição perfeitamente normal, mas sim reduzir a influência de valores extremos e produzir representações de atributos mais estáveis para os modelos.


In [25]:
df_features["Skewness"] = np.nan

numerical_features = (
    df_features.loc[
        df_features["Type"].eq("numerical"),
        "Name"
    ]
)

for feature in numerical_features:
    df_features.loc[
        df_features["Name"].eq(feature),
        "Skewness"
    ] = skew(df_full[feature].dropna())

A assimetria de cada variável numérica é calculada e armazenada na tabela de metadados dos atributos. Ordenar as variáveis pelo valor absoluto de sua assimetria permite identificar rapidamente aquelas cujas distribuições apresentam maiores desvios em relação à simetria e que, portanto, são candidatas à transformação logarítmica.


In [26]:
numerical_non_indicator_features = df_features.loc[
    df_features["Type"].eq("numerical")
    & ~df_features["Category"].eq("indicator"),
    "Name"
].tolist()

df_features.loc[df_features["Name"].isin(numerical_non_indicator_features),
    ["Name", "Category", "Skewness"]
].sort_values(
    "Skewness",
    ascending=False
).reset_index(drop=True)

,Name,Category,Skewness
0,MiscVal,original,21.947195
1,PoolArea,original,16.898328
2,LotArea,original,12.822431
3,LowQualFinSF,original,12.088761
4,3SsnPorch,original,11.376065
5,KitchenAbvGr,original,4.302254
6,BsmtFinSF2,original,4.146143
7,EnclosedPorch,original,4.003891
8,ScreenPorch,original,3.946694
9,BsmtHalfBath,original,3.931594


Embora a assimetria possa ser positiva ou negativa, transformações logarítmicas são principalmente eficazes na redução de assimetria positiva. Como as variáveis numéricas deste conjunto de dados são predominantemente assimétricas à direita, restringimos as transformações às variáveis cuja assimetria ultrapassa um limite positivo. Variáveis com assimetria negativa permanecem inalteradas, pois suas distribuições geralmente são menos problemáticas para os modelos considerados neste projeto.

Não existe um limite universal de assimetria a partir do qual uma transformação se torna necessária. Na prática, valores entre `0.5` e `1.0` são frequentemente utilizados como heurística para identificar variáveis com assimetria significativa.

Neste projeto, utilizamos um limite de **0.75**, que concentra a transformação nas variáveis com maior assimetria positiva, evitando modificações desnecessárias em atributos cuja distribuição apresenta apenas uma assimetria moderada.

Entretanto, uma alta assimetria por si só não é suficiente para justificar uma transformação logarítmica. Apenas atributos **numéricos e não indicadores** são considerados, pois aplicar uma transformação logarítmica em variáveis binárias oferece pouco benefício prático. Além disso, todos os valores de um atributo candidato devem ser maiores que **−1**, garantindo que a transformação `log1p` seja matematicamente definida e evitando valores indefinidos, como (\log(0)).

Somente os atributos que satisfazem todas essas condições são selecionados para a transformação.


In [27]:
# Conditions for log transformation
valid_features = df_features["Name"].isin(numerical_non_indicator_features)
log_safe_mask = df_full[numerical_non_indicator_features].min() > -1
SKEW_THRESHOLD = 0.75

# Assigning LogTransform flag
df_features["LogTransform"] = False
df_features.loc[
    valid_features &
    (df_features["Skewness"] > SKEW_THRESHOLD) &
    (df_features["Name"].map(log_safe_mask)),
    "LogTransform"
] = True

df_features

,Name,Type,Category,Source,Description,Skewness,LogTransform
0,MSSubClass,numerical,original,MSSubClass,Identifies the type of dwelling involved in th...,1.375457,True
1,MSZoning,categorical,original,MSZoning,Identifies the general zoning classification o...,NaN,False
2,LotFrontage,numerical,original,LotFrontage,Linear feet of street connected to property,0.022013,False
3,LotArea,numerical,original,LotArea,Lot size in square feet,12.822431,True
4,Street,categorical,original,Street,Type of road access to property,NaN,False
...,...,...,...,...,...,...,...
91,IsRemodeled,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has been remodeled,0.138046,False
92,HasPool,numerical,indicator,PoolArea,Indicates whether the property has a pool,14.884318,False
93,HasGarage,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has a garage,-3.941054,False
94,HasBasement,numerical,indicator,TotalBsmtSF,Indicates whether the property has a basement,-5.828995,False


## Codificação de variáveis categóricas

A maioria dos algoritmos de aprendizado de máquina requer entradas numéricas e não consegue processar diretamente variáveis categóricas representadas como rótulos de texto. Antes de treinar os modelos, portanto, precisamos converter os atributos categóricos para uma representação numérica.

Existem diversas técnicas de codificação, incluindo *label encoding*, *ordinal encoding*, *target encoding* e *one-hot encoding*. Neste projeto, utilizamos *one-hot encoding* por ser uma técnica simples, amplamente aplicável e que não impõe uma ordem artificial entre os valores categóricos.

A codificação *one-hot* cria uma variável binária para cada categoria presente em um atributo. Por exemplo, um atributo como `Neighborhood` é transformado em um conjunto de variáveis indicadoras, cada uma representando a presença ou ausência de um bairro específico.

Embora esse processo aumente o número de atributos no conjunto de dados, ele permite que os modelos de aprendizado de máquina utilizem informações categóricas de maneira consistente e interpretável.

As variáveis *dummy* geradas não são rastreadas individualmente na tabela de metadados `df_features`, pois são artefatos de implementação derivados automaticamente dos atributos categóricos originais, e não novos atributos conceituais criados durante a etapa de engenharia de atributos.

Primeiro identificamos os atributos categóricos registrados na tabela de metadados e, em seguida, aplicamos a codificação *one-hot* ao conjunto de dados.


In [28]:
categorical_features = df_features.loc[
    df_features["Type"].eq("categorical"),
    "Name"
]

df_full = pd.get_dummies(
    df_full,
    columns=categorical_features,
    dtype=int
)

## Salvar o conjunto de dados processado

Neste estágio, o conjunto de dados passou por todas as etapas de pré-processamento planejadas. Os valores ausentes foram tratados, novos atributos foram construídos, variáveis numéricas com alta assimetria foram transformadas quando apropriado, e os atributos categóricos foram codificados em uma representação numérica adequada para algoritmos de aprendizado de máquina.

Para evitar a repetição dessas etapas de pré-processamento no próximo notebook, salvamos tanto o conjunto de dados processado quanto a tabela de metadados de atributos atualizada. Dessa forma, a etapa de modelagem pode se concentrar exclusivamente no treinamento, avaliação e comparação dos modelos de regressão.

O conjunto de dados processado agora contém todas as informações necessárias para o desenvolvimento dos modelos e pode ser carregado diretamente no próximo notebook.


In [29]:
# Save dataframes in parquet format to preserve data types
df_full.to_parquet(
    "../data/processed/02_data.parquet",
    index=False
)
df_features.to_parquet(
    "../data/processed/02_features.parquet",
    index=False
)

## Resumo

Neste notebook, transformamos o conjunto de dados a partir da sua forma utilizada na análise exploratória em uma representação pronta para modelagem, utilizando uma combinação de técnicas de engenharia de atributos e pré-processamento.

As principais etapas realizadas foram:

* Criação de atributos agregados, como `TotalArea`, `TotalBathrooms` e `TotalPorchArea`.
* Construção de atributos baseados em tempo, como `HouseAge`, `YearsSinceRemodel` e `GarageAge`.
* Introdução de atributos de interação e razão, como `QualityArea`, `GarageAreaPerCar` e `LivingAreaPerRoom`.
* Criação de indicadores binários descrevendo a presença de características importantes dos imóveis.
* Combinação de determinadas comodidades no atributo `LuxuryFeatureCount`.
* Documentação de todos os atributos construídos através da tabela de metadados `df_features`.
* Análise da assimetria das variáveis numéricas e aplicação de transformações logarítmicas nas variáveis com alta assimetria positiva.
* Conversão das variáveis categóricas para uma representação numérica utilizando *one-hot encoding*.

O conjunto de dados resultante incorpora tanto conhecimento de domínio quanto técnicas de pré-processamento projetadas para melhorar o desempenho dos modelos e sua compatibilidade com algoritmos de aprendizado de máquina. Após essas transformações, o conjunto de dados é totalmente numérico e adequado para uma ampla variedade de modelos de regressão.

Para facilitar a próxima etapa do projeto, salvamos tanto o conjunto de dados processado quanto a tabela de metadados de atributos atualizada. Isso permite que o notebook de modelagem se concentre exclusivamente no treinamento, avaliação e comparação dos modelos, sem a necessidade de repetir o pipeline de pré-processamento.

No [próximo notebook](03_modeling_and_evaluation.ipynb), utilizaremos esse conjunto de dados transformado para treinar e avaliar diferentes modelos de regressão, comparar seus desempenhos e gerar uma submissão para a competição do Kaggle.
